# PySpark - Smart Lock Data Pipeline

Este notebook se conecta a las bases de datos del proyecto (PostgreSQL operacional y Redshift local) para consultar los datos generados por el simulador.

## Requisitos previos
- Docker Compose corriendo (`docker compose --env-file .env.aws up -d`)
- Datos cargados (haber corrido el simulador al menos una vez)

## 1. Crear la SparkSession

Usamos modo `local[*]` (todos los cores de tu maquina). No necesitas conectarte al cluster de Docker — para consultas interactivas es mas rapido y simple correr Spark localmente.

El driver JDBC de PostgreSQL se descarga automaticamente via `spark.jars.packages`.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SmartLock-Jupyter")
    .master("local[*]")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.1")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print(f"Spark UI: http://localhost:4040")

## 2. Conexion a PostgreSQL (operacional - RDS)

Esta es la base de datos operacional donde el consumer de Kafka escribe los datos.

In [ ]:
# --- Configuracion de conexion ---
# Para RDS (AWS):
POSTGRES_URL = "jdbc:postgresql://smartlock-db.cf5a6eemhoni.us-east-1.rds.amazonaws.com:5432/smartlock"
POSTGRES_USER = "smartlock"
POSTGRES_PASSWORD = "Smartlock2024poc"

# Para PostgreSQL local (si no usas RDS, descomentar estas lineas):
# POSTGRES_URL = "jdbc:postgresql://localhost:5432/smartlock"
# POSTGRES_USER = "smartlock"
# POSTGRES_PASSWORD = "smartlock"

# Redshift local (PostgreSQL en puerto 5439):
REDSHIFT_URL = "jdbc:postgresql://localhost:5439/smartlock_dw"
REDSHIFT_USER = "redshift"
REDSHIFT_PASSWORD = "redshift"

def read_table(table_name, url=POSTGRES_URL, user=POSTGRES_USER, password=POSTGRES_PASSWORD):
    """Lee una tabla completa como DataFrame de Spark."""
    return (
        spark.read
        .format("jdbc")
        .option("url", url)
        .option("dbtable", table_name)
        .option("user", user)
        .option("password", password)
        .option("driver", "org.postgresql.Driver")
        .load()
    )

def run_query(query, url=POSTGRES_URL, user=POSTGRES_USER, password=POSTGRES_PASSWORD):
    """Ejecuta un query SQL y devuelve un DataFrame."""
    return (
        spark.read
        .format("jdbc")
        .option("url", url)
        .option("query", query)
        .option("user", user)
        .option("password", password)
        .option("driver", "org.postgresql.Driver")
        .load()
    )

## 3. Explorar las tablas

In [ ]:
# Cargar todas las tablas
locks_df = read_table("locks")
events_df = read_table("security_events")
shipments_df = read_table("shipments")
assignments_df = read_table("lock_assignments")
locations_df = read_table("locations")

print("=== Conteo de registros ===")
print(f"Locks:            {locks_df.count()}")
print(f"Security Events:  {events_df.count()}")
print(f"Shipments:        {shipments_df.count()}")
print(f"Lock Assignments: {assignments_df.count()}")
print(f"Locations:        {locations_df.count()}")

In [ ]:
# Ver el schema de cada tabla
print("=== Locks ===")
locks_df.printSchema()
locks_df.show(5, truncate=False)

In [ ]:
print("=== Security Events ===")
events_df.printSchema()
events_df.show(5, truncate=False)

In [ ]:
print("=== Shipments ===")
shipments_df.show(truncate=False)

## 4. Consultas con la API de DataFrames

PySpark tiene dos formas de consultar datos:
- **API de DataFrames**: metodos encadenados (`.filter()`, `.groupBy()`, `.agg()`)
- **Spark SQL**: queries SQL directos

Ambos generan el mismo plan de ejecucion.

In [ ]:
from pyspark.sql import functions as F

# Eventos por tipo y severidad
events_df.groupBy("event_type", "severity").count().orderBy(F.desc("count")).show()

In [ ]:
# Locks por estado
locks_df.groupBy("status").count().show()

In [ ]:
# Top 5 locks con mas eventos de seguridad
(
    events_df
    .groupBy("lock_id")
    .agg(
        F.count("*").alias("total_events"),
        F.countDistinct("event_type").alias("distinct_types"),
        F.max("event_time").alias("last_event")
    )
    .orderBy(F.desc("total_events"))
    .show(5, truncate=False)
)

In [ ]:
# Locks con su ubicacion (JOIN)
(
    locks_df.alias("l")
    .join(locations_df.alias("loc"), F.col("l.id") == F.col("loc.lock_id"))
    .select("l.id", "l.status", "loc.latitude", "loc.longitude", "l.last_update")
    .show(10, truncate=False)
)

## 5. Consultas con Spark SQL

Registrando los DataFrames como vistas temporales, podes escribir SQL puro.

In [ ]:
# Registrar como vistas SQL
locks_df.createOrReplaceTempView("locks")
events_df.createOrReplaceTempView("security_events")
shipments_df.createOrReplaceTempView("shipments")
assignments_df.createOrReplaceTempView("lock_assignments")
locations_df.createOrReplaceTempView("locations")

In [ ]:
# Eventos criticos por shipment
spark.sql("""
    SELECT 
        s.shipment_id,
        COUNT(DISTINCT la.lock_id) AS total_locks,
        COUNT(se.id) AS total_events,
        SUM(CASE WHEN se.severity = 'critical' THEN 1 ELSE 0 END) AS critical_events
    FROM shipments s
    JOIN lock_assignments la ON la.shipment_id = s.id
    LEFT JOIN security_events se ON se.lock_id = la.lock_id
    GROUP BY s.shipment_id
    ORDER BY critical_events DESC
""").show(truncate=False)

In [ ]:
# Locks comprometidos (tampered/offline) con eventos recientes
spark.sql("""
    SELECT 
        l.id AS lock_id,
        l.status,
        COUNT(se.id) AS events_last_hour,
        COLLECT_SET(se.event_type) AS event_types
    FROM locks l
    LEFT JOIN security_events se 
        ON se.lock_id = l.id 
        AND se.event_time > current_timestamp() - INTERVAL 1 HOUR
    WHERE l.status IN ('tampered', 'offline')
    GROUP BY l.id, l.status
    ORDER BY events_last_hour DESC
""").show(truncate=False)

## 6. Consultar directamente con SQL (pushdown)

Con `run_query()` el SQL se ejecuta en PostgreSQL, no en Spark. Util para queries complejos donde PostgreSQL es mas eficiente, o para usar funciones especificas de PostgreSQL.

In [ ]:
# Este query se ejecuta en PostgreSQL directamente
result = run_query("""
    SELECT 
        event_type,
        severity,
        COUNT(*) as total,
        MIN(event_time) as first_event,
        MAX(event_time) as last_event
    FROM security_events
    GROUP BY event_type, severity
    ORDER BY total DESC
""")
result.show(truncate=False)

## 7. Consultar Redshift local

El data warehouse (Redshift emulado con PostgreSQL en puerto 5439) tiene las tablas de dimensiones y facts.

**Nota:** Estas tablas estaran vacias hasta que corras los jobs de PySpark streaming/batch.

In [ ]:
# Listar tablas en Redshift local
redshift_tables = run_query(
    "SELECT table_name FROM information_schema.tables WHERE table_schema = 'public' ORDER BY table_name",
    url=REDSHIFT_URL, user=REDSHIFT_USER, password=REDSHIFT_PASSWORD
)
redshift_tables.show(truncate=False)

## 8. Escribir resultados a Redshift

Podes transformar datos y escribirlos al data warehouse.

In [ ]:
# Ejemplo: crear un resumen y escribirlo a Redshift
summary = spark.sql("""
    SELECT
        lock_id,
        event_type,
        severity,
        COUNT(*) AS event_count,
        MIN(event_time) AS first_seen,
        MAX(event_time) AS last_seen
    FROM security_events
    GROUP BY lock_id, event_type, severity
""")

summary.show(5, truncate=False)

# Descomentar para escribir a Redshift local:
# summary.write \
#     .format("jdbc") \
#     .option("url", REDSHIFT_URL) \
#     .option("dbtable", "public.event_summary") \
#     .option("user", REDSHIFT_USER) \
#     .option("password", REDSHIFT_PASSWORD) \
#     .option("driver", "org.postgresql.Driver") \
#     .mode("overwrite") \
#     .save()
# print("Escrito a Redshift local.")

## 10. Limpiar

Siempre cerrar la SparkSession al terminar para liberar recursos.

### 9.1 row_number — Evento mas reciente por lock

`row_number()` asigna un numero secuencial dentro de cada particion. Util para obtener el "top N por grupo".

In [ ]:
from pyspark.sql.window import Window

# Ventana: particionar por lock_id, ordenar por event_time descendente
w_latest = Window.partitionBy("lock_id").orderBy(F.desc("event_time"))

# El evento mas reciente de cada lock (row_number = 1)
latest_event_per_lock = (
    events_df
    .withColumn("rn", F.row_number().over(w_latest))
    .filter(F.col("rn") == 1)
    .select("lock_id", "event_type", "severity", "event_time")
)

print("=== Evento mas reciente por lock ===")
latest_event_per_lock.show(10, truncate=False)

### 9.2 lag — Tiempo entre eventos consecutivos

`lag(col, n)` accede al valor de N filas atras dentro de la ventana. Permite comparar cada fila con la anterior.

In [ ]:
# Ventana: por lock_id, ordenado cronologicamente
w_chrono = Window.partitionBy("lock_id").orderBy("event_time")

# Calcular el tiempo entre cada evento y el anterior del mismo lock
time_between_events = (
    events_df
    .withColumn("prev_event_time", F.lag("event_time", 1).over(w_chrono))
    .withColumn("prev_event_type", F.lag("event_type", 1).over(w_chrono))
    .withColumn(
        "seconds_since_prev",
        (F.unix_timestamp("event_time") - F.unix_timestamp("prev_event_time"))
    )
    .select("lock_id", "event_type", "event_time", "prev_event_type", "seconds_since_prev")
    .filter(F.col("prev_event_time").isNotNull())  # excluir el primer evento de cada lock
)

print("=== Tiempo entre eventos consecutivos (por lock) ===")
time_between_events.show(15, truncate=False)

### 9.3 Running total — Acumulado de eventos por lock

Una ventana con `rowsBetween(Window.unboundedPreceding, 0)` significa "desde el inicio de la particion hasta la fila actual". Esto genera un **acumulado** (running total).

In [ ]:
# Ventana acumulativa: desde el inicio hasta la fila actual
w_running = (
    Window
    .partitionBy("lock_id")
    .orderBy("event_time")
    .rowsBetween(Window.unboundedPreceding, 0)  # todas las filas hasta la actual
)

# Conteo acumulado de eventos por lock
running_count = (
    events_df
    .withColumn("event_num", F.row_number().over(w_chrono))
    .withColumn("running_total", F.count("*").over(w_running))
    .withColumn(
        "running_critical",
        F.sum(F.when(F.col("severity") == "critical", 1).otherwise(0)).over(w_running)
    )
    .select("lock_id", "event_time", "event_type", "severity", "event_num", "running_total", "running_critical")
)

print("=== Acumulado de eventos por lock ===")
running_count.show(20, truncate=False)

### 9.4 rank vs dense_rank — Ranking de locks por cantidad de eventos

`rank()` deja huecos en empates (1, 2, 2, 4), `dense_rank()` no (1, 2, 2, 3). Aca no particionamos — rankeamos todos los locks juntos.

In [ ]:
# Primero agregamos: total de eventos por lock
events_per_lock = (
    events_df
    .groupBy("lock_id")
    .agg(F.count("*").alias("total_events"))
)

# Ventana global (sin partition) ordenada por total_events descendente
w_global_rank = Window.orderBy(F.desc("total_events"))

ranked_locks = (
    events_per_lock
    .withColumn("rank", F.rank().over(w_global_rank))
    .withColumn("dense_rank", F.dense_rank().over(w_global_rank))
    .orderBy("rank")
)

print("=== Ranking de locks por eventos (rank vs dense_rank) ===")
ranked_locks.show(15, truncate=False)

### 9.5 lead — Proximo evento (deteccion de patrones)

`lead(col, n)` mira N filas adelante. Util para detectar secuencias de eventos (ej: un `tamper_detected` seguido de `unlock_attempt` es un patron de ataque).

In [ ]:
# Ventana cronologica por lock
w_chrono = Window.partitionBy("lock_id").orderBy("event_time")

# Ver que evento viene despues de cada evento
event_sequences = (
    events_df
    .withColumn("next_event_type", F.lead("event_type", 1).over(w_chrono))
    .withColumn("next_event_time", F.lead("event_time", 1).over(w_chrono))
    .withColumn(
        "seconds_to_next",
        F.unix_timestamp("next_event_time") - F.unix_timestamp("event_time")
    )
    .select("lock_id", "event_type", "event_time", "next_event_type", "seconds_to_next")
    .filter(F.col("next_event_type").isNotNull())
)

print("=== Secuencias de eventos ===")
event_sequences.show(15, truncate=False)

# Detectar patron de ataque: tamper_detected seguido de unlock_attempt
print("=== Posibles ataques (tamper -> unlock) ===")
(
    event_sequences
    .filter(
        (F.col("event_type") == "tamper_detected") & 
        (F.col("next_event_type") == "unlock_attempt")
    )
    .show(truncate=False)
)

### 9.6 Ventana deslizante (sliding window) — Promedio movil

`rangeBetween` y `rowsBetween` definen el tamaño de la ventana:
- `rowsBetween(-2, 0)` = las 3 filas: 2 anteriores + la actual
- `rangeBetween(-3600, 0)` = todas las filas dentro de 1 hora antes de la actual (basado en el valor de la columna de orden)

In [ ]:
# Ventana deslizante por filas: las ultimas 3 filas (2 anteriores + actual)
w_sliding_3 = (
    Window
    .partitionBy("lock_id")
    .orderBy("event_time")
    .rowsBetween(-2, 0)  # 2 filas atras + fila actual = ventana de 3
)

# Conteo movil de eventos criticos en las ultimas 3 filas
sliding_critical = (
    events_df
    .withColumn("is_critical", F.when(F.col("severity") == "critical", 1).otherwise(0))
    .withColumn("critical_in_last_3", F.sum("is_critical").over(w_sliding_3))
    .withColumn("events_in_window", F.count("*").over(w_sliding_3))
    .select("lock_id", "event_time", "event_type", "severity", "events_in_window", "critical_in_last_3")
)

print("=== Ventana deslizante de 3 eventos ===")
sliding_critical.show(20, truncate=False)

### 9.7 Windows con Spark SQL

Todo lo anterior tambien se puede escribir en SQL puro. La sintaxis es:
```sql
funcion() OVER (PARTITION BY col ORDER BY col ROWS BETWEEN ... AND ...)
```

In [ ]:
# Mismo analisis que arriba pero en SQL puro
spark.sql("""
    SELECT
        lock_id,
        event_type,
        severity,
        event_time,

        -- row_number: numero secuencial
        ROW_NUMBER() OVER (PARTITION BY lock_id ORDER BY event_time) AS event_num,

        -- lag: evento anterior
        LAG(event_type, 1) OVER (PARTITION BY lock_id ORDER BY event_time) AS prev_event,

        -- lead: proximo evento
        LEAD(event_type, 1) OVER (PARTITION BY lock_id ORDER BY event_time) AS next_event,

        -- running total de eventos criticos
        SUM(CASE WHEN severity = 'critical' THEN 1 ELSE 0 END)
            OVER (PARTITION BY lock_id ORDER BY event_time
                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_critical,

        -- conteo en ventana deslizante de 3 filas
        COUNT(*) OVER (PARTITION BY lock_id ORDER BY event_time
                       ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS count_last_3

    FROM security_events
    ORDER BY lock_id, event_time
""").show(25, truncate=False)

## 9. Limpiar

Siempre cerrar la SparkSession al terminar para liberar recursos.

In [ ]:
# spark.stop()